# Tutorial: Garnet Port For Ariel Quantum Regression

Audience:
- Engineers working on `/Users/iwosmura/projects/hack4sages` who want one notebook to prepare validation runs for IQM Garnet from the saved Ariel checkpoint.

Prerequisites:
- Python 3.11 or 3.12
- The dependencies from `setup_garnet_env.sh`
- The saved checkpoint under `artifacts/ariel_quantum_best_v4_epoch6`
- The local Ariel dataset only when you want checkpoint-backed 8-qubit validation samples

Learning goals:
- Validate the local environment from inside the notebook
- Prepare 8-qubit Garnet-compatible validation circuits from Ariel samples
- Inspect dynamic layout selection and transpilation stats
- Run local and fake-backend dry validation
- Preview a real IQM validation submission without sending it


## Outline

1. Locate the repo and validate the Python environment
2. Import the Garnet helpers
3. Choose qubit count, backend mode, and validation split
4. Run preflight checks for checkpoint and dataset availability
5. Prepare either an 8-qubit validation run or a 12-qubit scaffold-only run
6. Inspect layout selection and transpilation details
7. Run local statevector and IQM fake-backend dry validation
8. Preview a real validation submission without sending it


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Could not find the repo root from the current working directory.')


REPO_ROOT_HINT = Path('/Users/iwosmura/projects/hack4sages')
REPO_ROOT = REPO_ROOT_HINT if (REPO_ROOT_HINT / '.git').exists() else find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


## Environment Check

This notebook is intended to be standalone inside the repo. The next cell checks that the key runtime dependencies are importable before any model code runs.


In [ ]:
required_modules = {
    'torch': 'checkpoint loading and frozen classical inference',
    'qiskit': 'circuit construction and transpilation',
    'iqm': 'IQM fake/mock/facade/real backend integration',
}
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    details = ', '.join(f"{name} ({required_modules[name]})" for name in missing)
    raise RuntimeError(
        'Missing required modules: ' + details + '. '        + f'Run {REPO_ROOT / "models" / "garnet_ariel_quantum_regression" / "setup_garnet_env.sh"} first.'
    )

from models.garnet_ariel_quantum_regression import (
    GarnetPortConfig,
    load_default_checkpoint_bundle,
    load_prepared_data,
    prepare_garnet_run,
    prepare_validation_split_run,
    resolve_split,
    run_iqm_execution,
    run_local_baseline,
    run_mock_evaluation,
)

'Environment looks ready.'


## Configuration

Use `N_QUBITS=8` for the saved checkpoint-backed validation path.

Use `N_QUBITS=12` only for scaffold, layout selection, and transpilation checks. The saved checkpoint is still 8-qubit, so 12-qubit prediction mode is intentionally blocked.


In [ ]:
N_QUBITS = 8
BACKEND_MODE = 'fake'   # 'fake', 'mock', 'facade', or 'real'
SPLIT_NAME = 'val'      # validation-only hardware path: 'val', 'holdout', or 'testdata'
MAX_SAMPLES = 4
SHOTS = 256
OPTIMIZATION_LEVEL = 1
SUBMIT_TO_IQM = False

config = GarnetPortConfig(
    n_qubits=N_QUBITS,
    backend_mode=BACKEND_MODE,
    shots=SHOTS,
    optimization_level=OPTIMIZATION_LEVEL,
    max_samples=MAX_SAMPLES,
    submit_to_iqm=SUBMIT_TO_IQM,
)
config


## Preflight Checks

This confirms the saved checkpoint exists. For 8-qubit validation, it also checks that the local Ariel dataset is available before trying to prepare samples.


In [ ]:
CHECKPOINT = load_default_checkpoint_bundle()
DATA_ROOT = REPO_ROOT / 'data' / 'ariel-ml-dataset'
checkpoint_ok = CHECKPOINT.checkpoint_dir.exists()
dataset_ok = DATA_ROOT.exists()

summary = {
    'checkpoint_dir': str(CHECKPOINT.checkpoint_dir),
    'checkpoint_exists': checkpoint_ok,
    'dataset_dir': str(DATA_ROOT),
    'dataset_exists': dataset_ok,
    'requested_qubits': N_QUBITS,
    'backend_mode': BACKEND_MODE,
}
print(json.dumps(summary, indent=2))

if N_QUBITS == 8 and not dataset_ok:
    raise RuntimeError('8-qubit validation mode needs the local Ariel dataset to prepare validation samples.')


## Load Validation Data Only When Needed

For `N_QUBITS=8`, the notebook loads a small prepared validation split.

For `N_QUBITS=12`, it skips dataset preparation entirely and stays in scaffold-only mode.


In [ ]:
prepared = load_prepared_data(data_root=DATA_ROOT, limit=MAX_SAMPLES) if N_QUBITS == 8 else None
data_split = resolve_split(prepared, SPLIT_NAME) if prepared is not None else None

if data_split is None:
    print('12-qubit scaffold mode: dataset loading skipped.')
else:
    print('rows =', data_split.rows)
    print('model_spectrum_shape =', prepared.split_manifest['model_spectrum_shape'])


## Prepare the Garnet Run

This is the single runtime path for the notebook:
- `prepare_validation_split_run(...)` for 8-qubit validation samples
- `prepare_garnet_run(..., data_split=None)` for 12-qubit scaffold-only preparation


In [ ]:
run = (
    prepare_validation_split_run(config, prepared, SPLIT_NAME)
    if N_QUBITS == 8
    else prepare_garnet_run(config, data_split=None)
)
summary = {
    'backend': run.backend_metadata,
    'layout': None if run.chosen_layout is None else {
        'logical_to_physical': run.chosen_layout.logical_to_physical,
        'ring_supported': run.chosen_layout.ring_supported,
        'degraded': run.chosen_layout.degraded,
        'reason': run.chosen_layout.reason,
    },
    'transpile_stats': run.transpile_stats,
}
print(json.dumps(summary, indent=2))


## Local Baseline and Fake-Backend Dry Validation

The local baseline uses exact statevector expectations.

The fake-backend evaluation uses transpiled measured circuits and shot counts, but it still stays local. This is the closest dry-run path before a real Resonance submission.


In [ ]:
if N_QUBITS != 8:
    print('Prediction cells are intentionally disabled for 12-qubit scaffold mode.')
else:
    local_predictions = run_local_baseline(run)
    mock_predictions = run_mock_evaluation(run)
    print('local_predictions.shape =', local_predictions.shape)
    print('mock_predictions.shape =', mock_predictions.shape)
    print('first_planet_id =', run.planet_ids[0])
    print('first_local_prediction =', local_predictions[0].tolist())


## Preview a Real IQM Validation Submission Without Sending It

`run_iqm_execution(...)` honors `submit_to_iqm=False` by default. In preview mode it only returns submission metadata from the prepared run and does not resolve or contact the live backend.

Switch to `BACKEND_MODE='real'` and `SUBMIT_TO_IQM=True` only when you are ready to send validation jobs to Resonance.


In [ ]:
submission_preview = run_iqm_execution(run)
submission_preview


## Pitfalls and Extensions

Common pitfalls:
- `N_QUBITS=12` is scaffold-only with the current saved checkpoint.
- Live hardware should be used only for validation, holdout, or testdata runs.
- `BACKEND_MODE='real'` still requires valid IQM access once `SUBMIT_TO_IQM=True`.

Useful extension:
- After you are comfortable with the validation flow, increase `MAX_SAMPLES` and compare `val` versus `holdout` transpilation statistics and prediction stability.


In [ ]:
# Optional inspection scaffold
if run.transpiled_circuits:
    first_circuit = run.transpiled_circuits[0]
    print(first_circuit)
else:
    print('No transpiled circuits are available.')
